# 03_Download Population and Built-Environment Data

This notebook uses local GHSL POP, GHSL BUILT-S, and WorldPop 2020 rasters to run city-level zonal stats for 86 city boundaries and generate population and built-environment control variables used by later Step 05/06/07 workflows.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd

ROOT = Path('/Volumes/ZHITAI2T/202606osm')
STEP_CODE = ROOT / 'code_upload/03_download_population_built_environment_data'
PROCESSOR_PATH = STEP_CODE / 'process_population_built_environment.py'

spec = importlib.util.spec_from_file_location('population_environment_processor', PROCESSOR_PATH)
processor = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = processor
assert spec.loader is not None
spec.loader.exec_module(processor)


## 1. Input Checks

Confirm that GHSL, WorldPop, and the 86 city boundaries are available.

In [ ]:
inventory = processor.build_local_inventory()
summary = processor.summarize_inventory(inventory)
coverage = processor.validate_worldpop_coverage()
plan = processor.build_city_processing_plan()

display(summary)
display(coverage['status'].value_counts())
display(plan['status'].value_counts())
display(plan[['city_id', 'city_name_en', 'iso3', 'status', 'dependency_reason']].head(20))


## 2. Write Processing Manifests

These manifests support review of input files, city coverage, and dependency status.

In [ ]:
prepare_outputs = processor.write_prepare_outputs()
prepare_outputs


## 3. Run Zonal Stats

The default setting is `all_touched=False`, which counts only pixels whose centers fall inside the city boundary.

In [ ]:
zonal_log = processor.write_zonal_stats_outputs(include_1000m=True, all_touched=False)
zonal_log


## 4. Output Checks

The primary output is `city_population_built_environment_metrics.parquet`; the CSV copy supports quick manual review.

In [ ]:
metrics_path = ROOT / 'data/03_download_population_built_environment_data/city_population_built_environment_metrics.parquet'
diagnostics_path = ROOT / 'data/03_download_population_built_environment_data/city_population_built_environment_zonal_stats_diagnostics.csv'

metrics = pd.read_parquet(metrics_path)
diagnostics = pd.read_csv(diagnostics_path)

key_cols = [
    'city_id', 'city_name_en', 'iso3', 'city_area_km2_calc',
    'ghsl_pop_2020_100m_sum', 'ghsl_built_s_2020_100m_sum', 'worldpop_pop_2020_sum',
    'ghsl_pop_density_2020_100m_per_km2', 'worldpop_pop_density_2020_per_km2',
    'ghsl_built_share_2020_100m', 'worldpop_to_ghsl_pop_2020_100m_ratio',
]

print(metrics.shape)
display(metrics[key_cols].head(20))
display(metrics[key_cols].isna().sum())
display(diagnostics['status'].value_counts())
